In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np
from pathlib import Path
from scipy.interpolate import interp1d

DRIVE = Path('/content/drive/MyDrive/airfoil_surrogate')
N_POINTS = 200
N_GRID = 100

In [ ]:
def extract_geom_features(X_raw, n_points=N_POINTS, n_grid=N_GRID):
    N = X_raw.shape[0]
    # Layout: [x0..x199, y0..y199, AoA, log_Re]
    coords = X_raw[:, :2*n_points].reshape(N, n_points, 2)
    x_grid = np.linspace(0.0, 1.0, n_grid)
    out = np.zeros((N, 2*n_grid + 5 + 2), dtype=np.float32)

    for i in range(N):
        xy = coords[i]
        x, y = xy[:, 0], xy[:, 1]
        le = int(np.argmin(x))
        h1x, h1y = x[:le+1][::-1], y[:le+1][::-1]
        h2x, h2y = x[le:],         y[le:]
        if h1y.mean() >= h2y.mean():
            xu, yu, xl, yl = h1x, h1y, h2x, h2y
        else:
            xu, yu, xl, yl = h2x, h2y, h1x, h1y

        def safe_interp(xs, ys):
            xs = np.clip(xs, 0, 1)
            o  = np.argsort(xs)
            xs, ys = xs[o], ys[o]
            _, u = np.unique(xs, return_index=True)
            xs, ys = xs[u], ys[u]
            if len(xs) < 3: return np.zeros(n_grid, np.float32)
            return interp1d(xs, ys, bounds_error=False,
                            fill_value=(ys[0], ys[-1]))(x_grid).astype(np.float32)

        y_u = safe_interp(xu, yu)
        y_l = safe_interp(xl, yl)
        t = y_u - y_l
        c = (y_u + y_l) / 2.0

        max_tc     = float(t.max())
        max_camber = float(np.abs(c).max())
        camber_loc = float(x_grid[np.argmax(np.abs(c))]) if max_camber > 0 else 0.5
        area       = float(np.trapz(t, x_grid))
        te_angle   = float(np.arctan(abs(t[-1] - t[-10]) /
                                     (x_grid[-1] - x_grid[-10] + 1e-8)))
        scalars = np.array([max_tc, max_camber, camber_loc, area, te_angle], np.float32)
        # Append AoA and log_Re from original array
        aoa_lre = X_raw[i, 2*n_points:].astype(np.float32)
        out[i] = np.concatenate([t, c, scalars, aoa_lre])

    return out  # (N, 209)

In [ ]:
import json
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive/airfoil_surrogate')

# Patch geom_input_dim to match actual array shape (207, not 209)
with open(DRIVE / 'dataset_stats.json') as f:
    stats = json.load(f)

stats['geom_input_dim'] = 207

with open(DRIVE / 'dataset_stats.json', 'w') as f:
    json.dump(stats, f, indent=2)

print('Patched geom_input_dim:', stats['geom_input_dim'])

import numpy as np
for split in ('train', 'val', 'test'):
    arr = np.load(DRIVE / f'X_geom_{split}.npy')
    print(f'X_geom_{split}: shape={arr.shape}, '
          f'mean={arr.mean():.4f}, std={arr.std():.4f}, '
          f'has_nan={np.isnan(arr).any()}, has_inf={np.isinf(arr).any()}')
    thickness_cols = arr[:, :100]
    print(f'  thickness: min={thickness_cols.min():.4f}, '
          f'max={thickness_cols.max():.4f} '
          f'(expect ~0 to ~0.5 for normalised airfoils)')
    camber_cols = arr[:, 100:200]
    print(f'  camber:    min={camber_cols.min():.4f}, '
          f'max={camber_cols.max():.4f} '
          f'(expect ~-0.1 to ~0.15)')

Patched geom_input_dim: 207
X_geom_train: shape=(6489, 207), mean=0.1089, std=0.7099, has_nan=False, has_inf=False
  thickness: min=-0.0000, max=0.6639 (expect ~0 to ~0.5 for normalised airfoils)
  camber:    min=-0.0137, max=0.0972 (expect ~-0.1 to ~0.15)
X_geom_val: shape=(1414, 207), mean=0.1097, std=0.7134, has_nan=False, has_inf=False
  thickness: min=-0.0000, max=0.2997 (expect ~0 to ~0.5 for normalised airfoils)
  camber:    min=-0.0066, max=0.1003 (expect ~-0.1 to ~0.15)
X_geom_test: shape=(1381, 207), mean=0.1060, std=0.7028, has_nan=False, has_inf=False
  thickness: min=-0.0000, max=0.2095 (expect ~0 to ~0.5 for normalised airfoils)
  camber:    min=-0.0110, max=0.1020 (expect ~-0.1 to ~0.15)
